In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import re
import lightgbm as lgb
import pickle
from datetime import datetime, timedelta

from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error

def ejecutar_todo(df):
    
    df = df.sort_values(['family', 'store_nbr', 'date'])
    df = df.reset_index(drop = True)
    
    df_inicial = df.copy()
    
    df['holiday_navidad'] = df.description.apply(lambda x: 1 if 'navidad' in str(x).lower() else 0)
    df['holiday_carnaval'] = df.description.apply(lambda x: 1 if 'carnaval' in str(x).lower() else 0)
    df['holiday_dia_padre_madre'] = df.description.apply(lambda x: 1 if 'dia de la madre' in str(x).lower() or 'dia del padre' in str(x).lower() else 0)
    df['holiday_cuenca'] = df.description.apply(lambda x: 1 if 'fundacion de cuenca' in str(x).lower() else 0)
    df['holiday_viernes_santo'] = df.description.apply(lambda x: 1 if 'viernes santo' in str(x).lower() else 0)
    df['holiday_primer_dia_ano'] = 0
    df.loc[(df.date.dt.month == 1) & (df.date.dt.day == 1), 'holiday_primer_dia_ano'] = 1
    
    df['day_of_week'] = df['date'].dt.day_of_week
    df['day_of_year'] = df['date'].dt.dayofyear
    df['day_of_month'] = df['date'].dt.day
    df['month'] = df['date'].dt.month
    df['quarter'] = df['date'].dt.quarter
    df['year'] = df['date'].dt.year
    
    df['terremoto'] = 0
    df.loc[(df.description.str.contains('Terremoto')) & (df.state == 'Manabi'), 'terremoto'] = 1
    
    df = df.drop(columns = ['tipo_holiday', 'locale', 'locale_name', 'description', 'transferred'])
    
    def salarios(x):
        if x.date.is_month_end or x.day_of_month == 15:
            return 1
        else: 
            return 0
    df['salarios'] = df.apply(salarios, axis = 1)
    
    df['abierta'] = 0
    for tienda in df['store_nbr'].unique():
        primer_dia_venta = df.loc[(df['store_nbr'] == tienda) & (df.sales > 0)].head(1).date.dt.strftime('%Y-%m-%d').iloc[0]
        df.loc[(df['store_nbr'] == tienda) & (df.date >= primer_dia_venta), 'abierta'] = 1
        
    df_agrupado = df.groupby(['store_nbr', 'family'])
    alphas = [0.95, 0.8, 0.65, 0.5]
    lags =[1,7,30]
    for a in alphas:
        for i in lags:
            df[f'sales_lag_{i}_alpha_{a}'] = np.log1p(df_agrupado['sales'].transform(lambda x: x.shift(i).ewm(alpha=a, min_periods=1).mean()))
            
    cat = ['city', 'state', 'tipo_store', 'store_nbr']
    df = pd.get_dummies(df, columns=cat, drop_first=True)
    
    sales_lag_columns = list(df.filter(like="lag").columns)
    df.dropna(inplace = True, subset = sales_lag_columns)
    
    with open('variables_finales.pickle', mode='rb') as file:
       variables_finales = pickle.load(file)
    
    variables_finales.remove('test')
    variables_finales.remove('id')
    
    df = df[variables_finales]

    with open('dict_modelos.pickle', mode='rb') as file:
       dict_modelos = pickle.load(file)
    
    lista_familias = df.family.unique()
    
    submission = pd.DataFrame()

    for familia in lista_familias:
        x = df.loc[df.family == familia].drop(columns=['date', 'family', 'sales'])
        modelo = dict_modelos[familia]

        predicciones = modelo.predict(x)
        predicciones_df = pd.DataFrame(data={'sales': predicciones}, index=x.index)

        submission = pd.concat([submission, predicciones_df], axis=0)
        
        patron = list(range(1, 55)) * 33
        store_nbr = pd.Series(patron[:1782])
        
    df_final = pd.merge(df_inicial[['date', 'family', 'store_nbr']], submission, left_index=True, right_index=True)
    df_final['sales'] = df_final['sales'].round(2)
    
    return df_final

In [2]:
path = r'C:\Users\Javier2\Desktop\Javier\Data Science\Proyectos\Forecasting_01'

nombre_fichero_datos = ['train.csv', 'stores.csv', 'oil.csv', 'holidays_events.csv', 'transactions.csv', 'test.csv', 'sample_submission.csv']

ruta_completa = path + '/02_Datos/01_Originales/' + nombre_fichero_datos[0]
train = pd.read_csv(ruta_completa, index_col = 0)
train['date'] = pd.to_datetime(train['date'])

ruta_completa = path + '/02_Datos/01_Originales/' + nombre_fichero_datos[1]
stores = pd.read_csv(ruta_completa)

ruta_completa = path + '/02_Datos/01_Originales/' + nombre_fichero_datos[2]
oil = pd.read_csv(ruta_completa)
oil['date'] = pd.to_datetime(oil['date'])

ruta_completa = path + '/02_Datos/01_Originales/' + nombre_fichero_datos[3]
holidays = pd.read_csv(ruta_completa)
holidays['date'] = pd.to_datetime(holidays['date'])

ruta_completa = path + '/02_Datos/01_Originales/' + nombre_fichero_datos[4]
transactions = pd.read_csv(ruta_completa)
transactions['date'] = pd.to_datetime(transactions['date'])

ruta_completa = path + '/02_Datos/01_Originales/' + nombre_fichero_datos[5]
test = pd.read_csv(ruta_completa)
test['date'] = pd.to_datetime(test['date'])

oil = oil.set_index('date')
oil = oil.resample('D').asfreq()
oil['precio'] = oil.dcoilwtico.rolling(6, min_periods = 2, center = True).mean()

holidays = holidays.drop_duplicates(subset = 'date', keep = 'last')

train = pd.merge(left = train, right = stores, how = 'left', on = 'store_nbr')
train = pd.merge(left = train, right = oil.reset_index(), how = 'left', on = 'date')
train = pd.merge(left = train, right = holidays, how = 'left', on = 'date')
train.drop_duplicates(subset = ['date', 'store_nbr', 'family'], keep = 'first', inplace = True)
train = train.reset_index(drop = True)
train = train.rename(columns = {'type_x':'tipo_store', 'type_y':'tipo_holiday'})
train['test'] = 0

test = pd.merge(left = test, right = stores, how = 'left', on = 'store_nbr')
test = pd.merge(left = test, right = oil.reset_index(), how = 'left', on = 'date')
test = pd.merge(left = test, right = holidays, how = 'left', on = 'date')
test.drop_duplicates(subset = ['date', 'store_nbr', 'family'], keep = 'first', inplace = True)
test = test.reset_index(drop = True)
test = test.rename(columns = {'type_x':'tipo_store', 'type_y':'tipo_holiday'})
test['test'] = 1

df = pd.concat([train, test], axis = 0)
df = df.loc[df.date >= '2017-07-01']
df

,date,store_nbr,family,sales,onpromotion,city,state,tipo_store,cluster,dcoilwtico,precio,tipo_holiday,locale,locale_name,description,transferred,test,id
2918916,2017-07-01,1,AUTOMOTIVE,7.0,0,Quito,Pichincha,D,13,NaN,45.213333,NaN,NaN,NaN,NaN,NaN,0,NaN
2918917,2017-07-01,1,BABY CARE,0.0,0,Quito,Pichincha,D,13,NaN,45.213333,NaN,NaN,NaN,NaN,NaN,0,NaN
2918918,2017-07-01,1,BEAUTY,7.0,1,Quito,Pichincha,D,13,NaN,45.213333,NaN,NaN,NaN,NaN,NaN,0,NaN
2918919,2017-07-01,1,BEVERAGES,2596.0,27,Quito,Pichincha,D,13,NaN,45.213333,NaN,NaN,NaN,NaN,NaN,0,NaN
2918920,2017-07-01,1,BOOKS,0.0,0,Quito,Pichincha,D,13,NaN,45.213333,NaN,NaN,NaN,NaN,NaN,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28507,2017-08-31,9,POULTRY,NaN,1,Quito,Pichincha,B,6,47.26,46.520000,NaN,NaN,NaN,NaN,NaN,1,3029395.0
28508,2017-08-31,9,PREPARED FOODS,NaN,0,Quito,Pichincha,B,6,47.26,46.520000,NaN,NaN,NaN,NaN,NaN,1,3029396.0
28509,2017-08-31,9,PRODUCE,NaN,1,Quito,Pichincha,B,6,47.26,46.520000,NaN,NaN,NaN,NaN,NaN,1,3029397.0
28510,2017-08-31,9,SCHOOL AND OFFICE SUPPLIES,NaN,9,Quito,Pichincha,B,6,47.26,46.520000,NaN,NaN,NaN,NaN,NaN,1,3029398.0


In [4]:
df.to_csv('df_app.csv', index=False) 